In [3]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

u = User(name="Omer", age="25")
print(u.age, type(u.age))


25 <class 'int'>


In [ ]:
User(name="Omer", age="twenty-five")
# ValidationError: Input should be a valid integer, unable to parse string as an integer

In [4]:
User(name="Omer", age="25.7")
# ValidationError — "25.7" isn't a clean int, no silent truncation

ValidationError: 1 validation error for User
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='25.7', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

In [5]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    price: float = Field(gt=0)              # must be > 0
    quantity: int = Field(ge=0, le=1000)     # 0 <= quantity <= 1000
    sku: str = Field(pattern=r"^[A-Z]{3}-\d{4}$")

Product(name="Mug", price=9.99, quantity=100, sku="MUG-0001")   # OK
Product(name="Mug", price=-5, quantity=100, sku="MUG-0001")
# ValidationError: Input should be greater than 0

ValidationError: 1 validation error for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

In [9]:
from pydantic import BaseModel, Field

class Product(BaseModel):
    name: str = Field(min_length=2, max_length=50)
    price: float = Field(gt=0)              # must be > 0
    quantity: int = Field(ge=0, le=1000)     # 0 <= quantity <= 1000
    sku: str = Field(pattern=r"^[A-Z]{3}-\d{4}$")

Product(name="Mug", price=9.99, quantity=100, sku="MUG-0001")   # OK
Product(name="Mug", price=-5, quantity=100, sku="MUG-0001")
# ValidationError: Input should be greater than 0

ValidationError: 1 validation error for Product
price
  Input should be greater than 0 [type=greater_than, input_value=-5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than

In [10]:
from typing import Optional
from pydantic import BaseModel

class Profile(BaseModel):
    username: str                      # required
    bio: Optional[str] = None          # optional, defaults to None
    followers: int = 0                 # optional, defaults to 0
    tags: list[str] = Field(default_factory=list)  # mutable defaults need default_factory

Profile(username="omer")
# username='omer' bio=None followers=0 tags=[]

Profile(username='omer', bio=None, followers=0, tags=[])

# Custom validators

In [8]:
from pydantic import BaseModel, field_validator, model_validator

class SignupForm(BaseModel):
    password: str
    confirm_password: str
    email: str

    @field_validator("email")
    @classmethod
    def email_must_be_lowercase(cls, v):
        if v != v.lower():
            raise ValueError("email must be lowercase")
        return v

    @model_validator(mode="after")   # runs after all fields are validated — needed for cross-field checks
    def passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("passwords do not match")
        return self

In [6]:
from pydantic import BaseModel

class Address(BaseModel):
    city: str
    zip_code: str

class Person(BaseModel):
    name: str
    address: Address        # nested model — Pydantic validates it recursively

p = Person(name="Omer", address={"city": "Lahore", "zip_code": "54000"})
print(p.address.city)   # Lahore — the dict got converted into a real Address object

Lahore


In [7]:
class Order(BaseModel):
    order_id: int
    items: list[Product]     # list of the Product model from section 1

order = Order(order_id=1, items=[
    {"name": "Mug", "price": 9.99, "quantity": 2, "sku": "MUG-0001"},
    {"name": "Plate", "price": 14.99, "quantity": 1, "sku": "PLT-0002"},
])
print(order.items[0].name)   # Mug

Mug


In [11]:
from enum import Enum
from typing import Literal, Union
from pydantic import BaseModel

class Role(str, Enum):
    ADMIN = "admin"
    USER = "user"

class Account(BaseModel):
    role: Role
    status: Literal["active", "suspended", "banned"]   # only these 3 strings allowed
    id: Union[int, str]    # accepts either type

Account(role="admin", status="active", id=42)      # OK
Account(role="owner", status="active", id=42)       # ValidationError — not a valid Role

ValidationError: 1 validation error for Account
role
  Input should be 'admin' or 'user' [type=enum, input_value='owner', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/enum